In [1]:
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.pytorch_utils import Conv1D

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

class LoRALinear(nn.Module):
    def __init__(self, base, r=8, alpha=16, dropout=0.0):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad = False
        d_in, d_out = base.weight.shape if isinstance(base, Conv1D) else (base.in_features, base.out_features)
        self.A = nn.Parameter(torch.randn(r, d_in) * 0.01)
        self.B = nn.Parameter(torch.zeros(d_out, r))
        self.scale = alpha / r
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.base(x) + ((self.dropout(x) @ self.A.T) @ self.B.T) * self.scale

def add_lora(model, r=8, alpha=16, dropout=0.05):
    for p in model.parameters():
        p.requires_grad = False
    for block in model.transformer.h:
        block.attn.c_attn = LoRALinear(block.attn.c_attn, r=r, alpha=alpha, dropout=dropout)
    return model

def quantize_4bit(w, block=64):
    flat = w.detach().flatten().float()
    pad = (-flat.numel()) % block
    if pad:
        flat = torch.cat([flat, torch.zeros(pad, device=flat.device)])
    blocks = flat.view(-1, block)
    scale = blocks.abs().amax(dim=1, keepdim=True).clamp(min=1e-8)
    q = torch.round(blocks / scale * 7).clamp(-8, 7).to(torch.int8)
    return q, scale, pad, w.shape

def dequantize(q, scale, pad, shape):
    out = (q.float() / 7 * scale).flatten()
    if pad:
        out = out[: out.numel() - pad]
    return out.view(shape)

stream_train = load_dataset("coastalcph/lex_glue", "ledgar", split="train", streaming=True)
names = stream_train.features["label"].names

sample_rows = list(stream_train.take(3000))
TOP = [l for l, _ in Counter(r["label"] for r in sample_rows).most_common(6)]
top_set = set(TOP)

train_rows = [r for r in sample_rows if r["label"] in top_set][:600]
train_ds = Dataset.from_list(train_rows)

stream_val = load_dataset("coastalcph/lex_glue", "ledgar", split="validation", streaming=True)
val_rows = [r for r in stream_val.take(1500) if r["label"] in top_set][:120]
val_ds = Dataset.from_list(val_rows)
trues = [names[e["label"]] for e in val_ds]

tok = AutoTokenizer.from_pretrained("distilgpt2")
tok.pad_token = tok.eos_token
tok.add_tokens([f"<{names[i]}>" for i in TOP])

label_to_token_id = {i: tok.convert_tokens_to_ids(f"<{names[i]}>") for i in TOP}
label_id_to_name = {v: names[k] for k, v in label_to_token_id.items()}
label_ids = list(label_id_to_name)

print(f"Device: {device} | Top 6 clauses: {[names[i] for i in TOP]}")
print(f"Train samples: {len(train_ds)} | Val samples: {len(val_ds)}")

/opt/conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
'[Errno -3] Temporary failure in name resolution' thrown while requesting GET https://huggingface.co/datasets/coastalcph/lex_glue/resolve/c23fdff1a6bf74e0e1a71cb86f1e781d37da888c/ledgar/validation-00000-of-00001.parquet
Retrying in 1s [Retry 1/5].
'[Errno -3] Temporary failure in name resolution' thrown while requesting GET https://huggingface.co/datasets/coastalcph/lex_glue/resolve/c23fdff1a6bf74e0e1a71cb86f1e781d37da888c/ledgar/validation-00000-of-00001.parquet
Retrying in 2s [Retry 2/5].


Device: cpu | Top 6 clauses: ['Governing Laws', 'Counterparts', 'Notices', 'Entire Agreements', 'Severability', 'Survival']
Train samples: 600 | Val samples: 120


In [2]:
import gc
torch.set_num_threads(2)

def train_tagger(quantize_base_first=False, epochs=2, lr=3e-3, batch_size=8):
    m = AutoModelForCausalLM.from_pretrained("distilgpt2", low_cpu_mem_usage=True)
    m.resize_token_embeddings(len(tok))
    add_lora(m)

    if quantize_base_first:
        with torch.no_grad():
            for mod in m.modules():
                if isinstance(mod, (Conv1D, nn.Linear)):
                    mod.weight.data = dequantize(*quantize_4bit(mod.weight.data))

    m.to(device)
    trainable = [p for p in m.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=lr)

    m.train()
    for epoch in range(epochs):
        shuffled = train_ds.shuffle(seed=epoch)
        total_loss = 0.0
        steps = 0
        for i in range(0, len(shuffled), batch_size):
            batch = shuffled[i : i + batch_size]
            texts = [t.strip()[:300] + " " for t in batch["text"]]
            targets = torch.tensor([label_to_token_id[l] for l in batch["label"]], device=device)

            enc = tok(texts, padding=True, truncation=True, max_length=64, return_tensors="pt").to(device)
            last_pos = enc["attention_mask"].sum(dim=1) - 1
            hidden = m.transformer(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"]).last_hidden_state
            pred_logits = m.lm_head(hidden[torch.arange(len(texts), device=device), last_pos])

            loss = F.cross_entropy(pred_logits, targets)
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            steps += 1
            if steps % 25 == 0:
                print(f"  Epoch {epoch + 1} | Step {steps}/{len(shuffled) // batch_size} | Loss: {total_loss / steps:.4f}", flush=True)
        print(f"Epoch {epoch + 1}/{epochs} | Avg Loss: {total_loss / steps:.4f}", flush=True)

    m.to("cpu").eval()
    return m

fp32_tagger = train_tagger(quantize_base_first=False, epochs=2)

adapter_state = {k: v for k, v in fp32_tagger.state_dict().items() if "A" in k or "B" in k}
adapter_state["label_embeddings"] = fp32_tagger.transformer.wte.weight[-len(TOP):].detach().clone()
torch.save(adapter_state, "lora-adapter.pt")

del fp32_tagger
gc.collect()
print("Saved LoRA adapter to lora-adapter.pt")

Loading weights: 100%|██████████| 76/76 [00:00<00:00, 142.98it/s]
[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


  Epoch 1 | Step 25/75 | Loss: 12.3803
  Epoch 1 | Step 50/75 | Loss: 11.6783
  Epoch 1 | Step 75/75 | Loss: 11.4293
Epoch 1/2 | Avg Loss: 11.4293
  Epoch 2 | Step 25/75 | Loss: 10.9170
  Epoch 2 | Step 50/75 | Loss: 10.9140
  Epoch 2 | Step 75/75 | Loss: 10.9110
Epoch 2/2 | Avg Loss: 10.9110
Saved LoRA adapter to lora-adapter.pt
